# Quantification of Mitochondria in 2D Fluorescent Stacks
- *Isobel Taylor-Hearn, 2023*
- Requires a fluorescent image stack containing (at least) a nuclear channel and mitochondria channel

In [10]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import math
from cellpose import models
import os
import pathlib
from collections import Counter
from skimage.segmentation import find_boundaries
from liffile import LifFile
from skimage.feature import peak_local_max
from skimage import util,  restoration
from skimage.filters import threshold_otsu, threshold_isodata, threshold_mean, gaussian, threshold_yen, threshold_minimum, threshold_triangle
from skimage.segmentation import clear_border, expand_labels, watershed
from skimage.measure import label, regionprops_table
from skimage.morphology import remove_small_holes, remove_small_objects, closing,  disk, dilation
from scipy.ndimage import distance_transform_edt
from scipy import ndimage as ndi
from tifffile import imread
from skimage.filters import try_all_threshold
import numpy as np
from skimage import exposure
import warnings
warnings.filterwarnings("ignore")
import pathlib

import contextlib
import joblib
from tqdm import tqdm
from joblib import Parallel, delayed

In [11]:
@contextlib.contextmanager
def tqdm_joblib(tqdm_object):
    """
    Enables parallel jobs to run and display of a tqdm progress bar.

    Parameters:
    -----------
    tqdm_object : tqdm
        The tqdm progress bar instance to be updated.

    """
    class TqdmBatchCompletionCallback(joblib.parallel.BatchCompletionCallBack):
        def __call__(self, *args, **kwargs):
            tqdm_object.update(n=self.batch_size)
            return super().__call__(*args, **kwargs)

    old_batch_callback = joblib.parallel.BatchCompletionCallBack
    joblib.parallel.BatchCompletionCallBack = TqdmBatchCompletionCallback
    try:
        yield tqdm_object
    finally:
        joblib.parallel.BatchCompletionCallBack = old_batch_callback
        tqdm_object.close()

In [12]:
def convert(img, target_type_min, target_type_max, target_type):
    """
    Converts an image to a specified data type while scaling its intensity values.

    This function rescales the intensity values of an image from its original range 
    to a new target range specified by `target_type_min` and `target_type_max`, and 
    then converts it to the desired data type.

    This step is required as deconvolved images are not always scaled 0->255! 

    Parameters:
    -----------
    img : numpy.ndarray
        The input image array to be converted.
    target_type_min : int or float
        The minimum value of the target intensity range.
    target_type_max : int or float
        The maximum value of the target intensity range.
    target_type : numpy.dtype
        The desired data type of the output image (e.g., np.uint8, np.float32).

    Returns:
    --------
    new_img : numpy.ndarray
        The rescaled image with values mapped to the new intensity range and converted 
        to the specified data type.

    Notes:
    ------
    - This function performs a linear transformation to scale pixel values.
    - It ensures that the output values are properly mapped between `target_type_min` and 
      `target_type_max`.
    """
    imin = img.min()
    imax = img.max()

    a = (target_type_max - target_type_min) / (imax - imin)
    b = target_type_max - a * imax
    new_img = (a * img + b).astype(target_type)
    return new_img

In [13]:
def add_image_details(df, filename):
    """
    Adds experimental details extracted from the filename to a dataframe.

    This function parses the filename to infer experimental details such as 
    well number, imaging day, mechanical stiffness condition, and treatment type.
    The extracted details are appended as new columns to the dataframe.

    Parameters:
    -----------
    df : pandas.DataFrame
        The dataframe to which image metadata will be added.
    filename : str
        The filename of the image, used to extract experimental details.

    Returns:
    --------
    df : pandas.DataFrame
        The updated dataframe with the following added columns:
        - 'filename': The original filename.
        - 'cell_type': L_LNS or M_LNS
        - 'treatment': Blebbistatin, ROCKi, or untreated
        - 'timepoint': 0hr, 1hr, or 24hr
    """

    df["filename"] = filename
    test_filename = str(filename).lower()
    if "e2p4" in test_filename and "tnf" in test_filename and "fulv" in test_filename:
        df["treatment"] = "E2P4_TNF_FULV" 
        
    elif "e2p4" in test_filename and "tnf" in test_filename:
        df["treatment"] = "E2P4_TNF"
    elif "fulv" in test_filename and  "tnf" in test_filename:
        df["treatment"] = "FULV_TNF"
        
    elif "ctrl" in test_filename:
        df["treatment"] = "NONE"
    elif "ep24" in test_filename:
        df["treatment"] = "E2P4"  
    elif "tnf" in test_filename:
        df["treatment"] = "TNF"

    else:
        df["treatment"] = "UNKNOWN!"  


    return df

In [14]:
def segment_nuclei(dapi, protein_of_interest_1, phalloidin, model=None):
    """
    Segment cells and nuclei with Cellpose.

    Cells are segmented from a 2-channel image: a fused cell body = max(phalloidin, ICAM1)
    (both fill the cytoplasm) plus DAPI, which lets Cellpose use the nuclei to split
    touching cells. Nuclei are segmented separately from DAPI by thresholding + watershed.

    Parameters
    ----------
    dapi : ndarray
        Nuclear (DAPI) image.
    protein_of_interest_1 : ndarray
        ICAM1 image (cytoplasmic).
    phalloidin : ndarray
        Phalloidin image - edges + cytoplasm.
    model : cellpose model, optional
        A pre-built CellposeModel to reuse (avoids reloading weights every image). If None,
        a new one is created.

    Returns
    -------
    dapi_labels : ndarray (int)
        Segmented nuclei.
    all_cells : ndarray (int)
        Segmented cells.
    """
    if model is None:
        model = models.CellposeModel(gpu=True, pretrained_model="cpsam_v2")

    def _n(a):
        a = a.astype(np.float32)
        p1, p2 = np.percentile(a, [1, 99.7])
        return np.clip((a - p1) / (p2 - p1), 0, 1) if p2 > p1 else np.zeros_like(a)

    body = np.maximum(_n(phalloidin), _n(protein_of_interest_1))  # phalloidin + ICAM1 fill the cell
    cell_input = np.stack([body, _n(dapi)], axis=-1)   # [cell body, DAPI]
    all_cells, _, _ = model.eval(cell_input, channel_axis=-1, normalize=True,
                                 min_size=50, flow_threshold=0.6, cellprob_threshold=-2.0)
    all_cells = np.asarray(all_cells)

    dapi_processed = dapi > threshold_otsu(dapi)
    dapi_processed = remove_small_holes(dapi_processed, area_threshold=20)
    dapi_processed = remove_small_objects(dapi_processed, min_size=50)
    dapi_processed = closing(dapi_processed, disk(3))
    dapi_labels = label(dapi_processed)
    distances = ndi.distance_transform_edt(dapi_processed)
    coordinates = peak_local_max(distances, min_distance=10)  # min separation of nuclear COMs
    marker_locations = coordinates.data
    markers = np.zeros(dapi_processed.shape, dtype=np.uint32)
    marker_indices = tuple(np.round(marker_locations).astype(int).T)
    markers[marker_indices] = np.arange(len(marker_locations)) + 1

    dapi_labels_ws = watershed(-distances, markers, mask=dapi_processed)
    dapi_labels = label(dapi_labels_ws + dapi_labels)

    return dapi_labels, all_cells

In [15]:

def lif_pixel_sizes(img):
    """Return (z_um, y_um, x_um) from a liffile image; any value is None if unavailable."""
    z_um = y_um = x_um = None
    try:
        coords = img.asxarray().coords
        if "X" in coords and coords["X"].size >= 2:
            x_um = abs(float(coords["X"][1] - coords["X"][0])) * 1e6
        if "Y" in coords and coords["Y"].size >= 2:
            y_um = abs(float(coords["Y"][1] - coords["Y"][0])) * 1e6
        if "Z" in coords and coords["Z"].size >= 2:
            z_um = abs(float(coords["Z"][1] - coords["Z"][0])) * 1e6
    except Exception as e:
        print(f"  [WARN] could not read LIF pixel sizes: {e}")
    return z_um, y_um, x_um

In [16]:
def preprocess_dim_fluorescence(image, background_sigma=10, denoise_sigma=1, low_percentile=1, high_percentile=99.8,):
    img = image.astype(np.float32)
    background = gaussian(img, sigma=background_sigma)
    img_corrected = img - background
    img_corrected[img_corrected < 0] = 0
    img_smooth = gaussian(img_corrected, sigma=denoise_sigma)
    p_low, p_high = np.percentile(img_smooth, [low_percentile, high_percentile])
    img_rescaled = exposure.rescale_intensity(img_smooth, in_range=(p_low, p_high), out_range=(0, 1))
    return img_rescaled

In [ ]:
import threading

# Only one thread runs GPU inference at a time; CPU pre/post-processing of other images overlaps.
GPU_LOCK = threading.Lock()


def plot_segmentation(output_dir, filename, rgb_image, cyto_signal, gap_labels, gap_area_pct,
                       dapi_out, dapi_labels, protein_of_interest_2, cell_labels_paired, dapi_labels_paired,
                       n_nuclei):
    """Draw + save the 6-panel segmentation figure for one image (main thread only)."""
    fig, ax = plt.subplots(ncols=6, figsize=(20, 5))
    ax[0].imshow(rgb_image)
    ax[0].set_title(filename, fontsize=8)

    ax[1].imshow(cyto_signal, cmap="gray", interpolation="none")
    ax[1].imshow(np.ma.masked_where(gap_labels == 0, gap_labels), cmap="jet", interpolation="none", alpha=0.8)
    ax[1].set_title(f"gaps {gap_area_pct:.1f}%")

    ax[2].imshow(dapi_out, cmap="gray", interpolation="none")
    ax[2].imshow(np.ma.masked_where(dapi_labels == 0, dapi_labels), cmap="jet", interpolation="none", alpha=0.5)
    ax[2].set_title(f"all nuclei (n={n_nuclei})")

    ax[3].imshow(protein_of_interest_2, cmap="gray")

    cell_boundaries = find_boundaries(cell_labels_paired, mode="outer")
    ax[4].imshow(np.ma.masked_where(cell_labels_paired == 0, cell_labels_paired), cmap="jet", alpha=0.3, interpolation="none")
    ax[4].imshow(np.ma.masked_where(~cell_boundaries, cell_labels_paired), cmap="gray")
    ax[4].set_title("paired cells")
    ax[5].imshow(protein_of_interest_2, cmap="gray")
    ax[5].imshow(np.ma.masked_where(cell_labels_paired == 0, cell_labels_paired), cmap="jet", interpolation="none")
    ax[5].imshow(np.ma.masked_where(~cell_boundaries, cell_labels_paired), cmap="gray")
    ax[5].imshow(np.ma.masked_where(dapi_labels_paired == 0, dapi_labels_paired), cmap="Reds", interpolation="none")
    ax[5].set_title("paired cells + nuclei")

    for a in ax:
        a.axis("off")
    plt.savefig(fr"{output_dir}/{filename}_segmentation.png", bbox_inches="tight")
    plt.show()


def process_one_image(data, z_um, y_um, x_um, filename, model, to_plot, protein_of_interest_2_name, protein_of_interest_2_channel,
                       dapi_channel, protein_of_interest_1_name, protein_of_interest_1_channel,
                     phalloidin_channel, pixel_size, max_proj):
    """Segment + measure a single image. Returns (per_cell_df, image_summary_df, plot_data|None)."""
    if max_proj == True:
        dapi = convert(data[:,dapi_channel, :, :].sum(axis=0), 0, 255, np.uint8)
        protein_of_interest_1 = convert(data[:,protein_of_interest_1_channel, :, :].sum(axis=0), 0, 255, np.uint8)
    else:
        dapi = convert(data[:,dapi_channel, :, :], 0, 255, np.uint8)
        protein_of_interest_1 = convert(data[:,protein_of_interest_1_channel, :, :], 0, 255, np.uint8)

    if protein_of_interest_2_channel is not None:
        if max_proj == True:
            protein_of_interest_2 = convert(data[:,protein_of_interest_2_channel, :, :].max(axis=0), 0, 100, np.uint8)
        else:
            protein_of_interest_2 = convert(data[:,protein_of_interest_2_channel, :, :].sum(axis=0), 0, 255, np.uint8)

    dapi_out = preprocess_dim_fluorescence(dapi)

    if phalloidin_channel is not None:
        if max_proj == True:
            phalloidin = convert(data[phalloidin_channel, :, :].max(axis=0), 0, 255, np.uint8)
        else:
            phalloidin = convert(data[phalloidin_channel, :, :], 0, 255, np.uint8)
        rgb_image = np.stack([protein_of_interest_1, phalloidin, dapi_out], axis=-1)
    elif protein_of_interest_2_channel is not None:
        rgb_image = np.stack([np.zeros_like(dapi), protein_of_interest_2, dapi_out], axis=-1)
    else:
        rgb_image = np.stack([protein_of_interest_1, protein_of_interest_2, dapi], axis=-1)

    # Physical area per pixel (fall back to square pixels if only one size known).
    pixel_area_um2 = (x_um * y_um) if (x_um and y_um) else (pixel_size ** 2)

    with GPU_LOCK:                                   # serialise GPU inference across threads
        dapi_labels, cell_labels = segment_nuclei(dapi_out, protein_of_interest_1, phalloidin, model=model)

    # Tissue signal from the cytoplasm-filling markers, put on a common 0-1 scale.
    cyto_signal = np.maximum(protein_of_interest_1.astype(np.float32) / 255,
                                phalloidin.astype(np.float32) / 255)
    cyto_signal = gaussian(np.clip(cyto_signal, a_min=0, a_max=np.quantile(cyto_signal, 0.7)), sigma=2)
    try:
        cells = (cyto_signal > threshold_minimum(cyto_signal))
    except:
        cells = (cyto_signal > threshold_triangle(cyto_signal))

    cells = remove_small_holes(cells, area_threshold=50)
    cells = remove_small_objects(cells, min_size=200)

    # Enclosed background only (excludes the exterior around the monolayer).
    monolayer = ndi.binary_fill_holes(cells)
    gaps = monolayer & ~cells
    gaps = remove_small_objects(gaps, min_size=50)           # min gap size (pixels)

    gap_labels = label(gaps)
    gap_props = pd.DataFrame(regionprops_table(gap_labels, properties=("label", "area")))
    gap_props["gap_area_um2"] = gap_props["area"] * pixel_area_um2
    n_gaps = len(gap_props)
    total_gap_um2 = float(gap_props["gap_area_um2"].sum())
    gap_area_pct = 100.0 * gaps.sum() / monolayer.sum()      # % of monolayer that is gap
    print(f"{filename}: {n_gaps} gaps, {total_gap_um2:.0f} um^2 total, {gap_area_pct:.1f}% of monolayer")

    # --- Per-object shape/size + ICAM1 intensity ---
    cell_measure = pd.DataFrame(regionprops_table(
        cell_labels, intensity_image=protein_of_interest_1,
        properties=("label", "area", "eccentricity", "solidity", "intensity_mean")))
    cell_measure = cell_measure.rename(columns={
        "label": "cell_label", "area": "cell_area_px",
        "eccentricity": "cell_eccentricity", "solidity": "cell_solidity",
        "intensity_mean": f"{protein_of_interest_1_name}_cell_mean"})

    nuc_measure = pd.DataFrame(regionprops_table(
        dapi_labels, intensity_image=protein_of_interest_1,
        properties=("label", "area", "eccentricity")))
    nuc_measure = nuc_measure.rename(columns={
        "label": "nucleus_label", "area": "nuclear_area_px",
        "eccentricity": "nuclear_eccentricity"})

    # Pair each nucleus with the cell it overlaps most; drop nuclei with no cell.
    flat = pd.DataFrame({"nucleus_label": dapi_labels.ravel(),
                            "cell_label": cell_labels.ravel()})
    flat = flat[flat["nucleus_label"] > 0]
    paired = (flat.groupby("nucleus_label")["cell_label"]
                    .agg(lambda s: s.value_counts().idxmax()).reset_index())
    paired = paired[paired["cell_label"] > 0]

    props = (paired.merge(nuc_measure, on="nucleus_label")
                    .merge(cell_measure, on="cell_label"))

    # Per-cell + per-nucleus intensity (needs both cell_label and nucleus_label, so this
    # runs after pairing, not on cell_measure alone).
    if protein_of_interest_2_channel is not None:
        props[f"{protein_of_interest_2_name}_cell"] = props.apply(lambda row: protein_of_interest_2[cell_labels == row["cell_label"]].sum(), axis = 1)
        props[f"{protein_of_interest_2_name}_nucleus"] = props.apply(lambda row: protein_of_interest_2[(cell_labels == row["cell_label"]) & (dapi_labels == row["nucleus_label"])].sum(), axis = 1) 
        props[f"{protein_of_interest_2_name}_nucleus/cell"] = props[f"{protein_of_interest_2_name}_nucleus"]/props[f"{protein_of_interest_2_name}_cell"]
    if protein_of_interest_1_channel is not None:
        props[f"{protein_of_interest_1_name}_cell"] = props.apply(lambda row: protein_of_interest_1[cell_labels == row["cell_label"]].sum(), axis = 1)
        props[f"{protein_of_interest_1_name}_nucleus"] = props.apply(lambda row: protein_of_interest_1[(cell_labels == row["cell_label"]) & (dapi_labels == row["nucleus_label"])].sum(), axis = 1) 
        props[f"{protein_of_interest_1_name}_nucleus/cell"] = props[f"{protein_of_interest_1_name}_nucleus"]/props[f"{protein_of_interest_1_name}_cell"]

    # Drop cells smaller than their paired nucleus.
    props = props[props["cell_area_px"] >= props["nuclear_area_px"]].copy()
    # Drop cells touching the image border (and their paired nuclei).
    border_cells = np.setdiff1d(np.unique(cell_labels), np.unique(clear_border(cell_labels)))
    props = props[~props["cell_label"].isin(border_cells)].copy()

    props["nuclear_area_um2"] = props["nuclear_area_px"] * pixel_area_um2
    props["cell_area_um2"] = props["cell_area_px"] * pixel_area_um2
    props["nuc/cell_area"] = props["nuclear_area_px"] / props["cell_area_px"]


    # Per-cell table: which image / treatment each measured cell came from.
    props["filename"] = filename
    props = add_image_details(props, filename)

    # Per-image overview (assume 1 nucleus : 1 cell; the monolayer-minus-gap area is shared
    # equally between cells).
    n_nuclei = int(np.unique(dapi_labels[dapi_labels > 0]).size)
    monolayer_area_um2 = float(monolayer.sum()) * pixel_area_um2
    total_cell_area_um2 = monolayer_area_um2 - float(gaps.sum()) * pixel_area_um2
    mean_cell_area_assumed_um2 = total_cell_area_um2 / n_nuclei if n_nuclei else np.nan
    img_summary = add_image_details(pd.DataFrame([{
        "gap_area_pct": gap_area_pct,
        "n_nuclei_fov": n_nuclei,
        "n_cells_assumed": n_nuclei,
        "mean_cell_area_assumed_um2": mean_cell_area_assumed_um2,
        "total_cell_area_um2": total_cell_area_um2,
        "monolayer_area_um2": monolayer_area_um2,
        "total_gap_um2": total_gap_um2,
        "n_gaps": n_gaps,
    }]), filename)

    plot_data = None
    if to_plot:
        kept_cells = props["cell_label"].to_numpy()
        kept_nuclei = props["nucleus_label"].to_numpy()
        plot_data = dict(
            filename=filename, rgb_image=rgb_image, cyto_signal=cyto_signal,
            gap_labels=gap_labels, gap_area_pct=gap_area_pct, dapi_out=dapi_out,
            dapi_labels=dapi_labels, protein_of_interest_2=protein_of_interest_2, n_nuclei=n_nuclei,
            cell_labels_paired=np.where(np.isin(cell_labels, kept_cells), cell_labels, 0),
            dapi_labels_paired=np.where(np.isin(dapi_labels, kept_nuclei), dapi_labels, 0))

    return props, img_summary, plot_data


def segment_nuclei_and_quantify_protein(input_directory,  output_dir , to_plot = True, dapi_channel = 0, protein_of_interest_2_channel=None, protein_of_interest_2_name="VECad", protein_of_interest_1_channel = 1, protein_of_interest_1_name="ICAM1", phalloidin_name=None, phalloidin_channel = None, pixel_size = 7.4627, max_proj = False, n_jobs = 4):
    """
    Segment cells + nuclei from a multi-channel image and quantify a protein of interest per cell.

    Individual images within the LIF(s) are processed in parallel across `n_jobs` threads that
    share one Cellpose model; GPU inference is serialised (only the CPU work overlaps). Set
    n_jobs=1 for fully sequential processing.

    Returns
    -------
    per_cell_props : pandas.DataFrame -> this is just the cells we're "sure" about -> matched nucleus and cell, not touching image border
        One row per measured (paired, non-border) cell, including:
        - 'filename', 'treatment'
        - 'label'
        - 'cell_area_um2'      : cell area in square microns
        - 'nuclear_area_um2'   : nucleus area in square microns
        - 'nuc/cell_area'      : nuclear / cell area ratio
        - '{protein}_cell', '{protein}_nucleus', '{protein}_nucleus/cell' intensity measures
    image_summary : pandas.DataFrame -> use this to approximate cell area + gap %
        One row per image: 'gap_area_pct', 'n_nuclei_fov', 'n_cells_assumed' (= n nuclei,
        1:1), 'mean_cell_area_assumed_um2' ((monolayer - gap) area / n nuclei),
        'total_cell_area_um2', 'monolayer_area_um2', 'total_gap_um2', 'n_gaps'.
    """
    # One shared model across all images (avoids reloading weights / multiple GPU copies).
    model = models.CellposeModel(gpu=True, pretrained_model="cpsam_v2")

    # Read every image up front in the main process (LifFile handles aren't thread-friendly).
    tasks = []
    for lif_path in sorted(input_directory.rglob("*.lif")):
        print(f"LIF: {lif_path}")
        with LifFile(lif_path) as lif:
            for img in lif.images:
                sub_name = "".join(img.path)
                data = img.asarray()
                z_um, y_um, x_um = lif_pixel_sizes(img)
                filename = f"{lif_path.stem}_{sub_name}".replace(" ", "")
                print(filename, data.shape, (z_um, y_um, x_um))
                tasks.append((data, z_um, y_um, x_um, filename))

    # Segment + measure images in parallel (threads; GPU inference serialised by GPU_LOCK).
    with tqdm_joblib(tqdm(desc="Segmenting images", total=len(tasks))):
        results = Parallel(n_jobs=n_jobs, backend="threading")(
            delayed(process_one_image)(
                data, z_um, y_um, x_um, filename, model, to_plot, protein_of_interest_2_name, protein_of_interest_2_channel,
                dapi_channel, protein_of_interest_1_name, protein_of_interest_1_channel,
                phalloidin_channel, pixel_size, max_proj)
            for (data, z_um, y_um, x_um, filename) in tasks)

    per_cell_props = [r[0] for r in results]
    image_summaries = [r[1] for r in results]

    # Plot in the main thread (matplotlib is not thread-safe).
    if to_plot:
        for _, _, plot_data in results:
            if plot_data is not None:
                plot_segmentation(output_dir, **plot_data)

    per_cell = pd.concat(per_cell_props, ignore_index=True) if per_cell_props else pd.DataFrame()
    image_summary = pd.concat(image_summaries, ignore_index=True) if image_summaries else pd.DataFrame()
    return per_cell, image_summary

In [ ]:
input_dir = pathlib.Path(r"Z:\Bel\Dorota_Image_Analzsis_Size_Spaces")
output_dir = pathlib.Path(f"{str(input_dir)}\outputs_new")
output_dir.mkdir(parents=True, exist_ok=True)
dapi_channel = 0
protein_of_interest_2_channel = 2
protein_of_interest_channel = 1
phalloidin_channel = 3 
per_cell_props, per_image = segment_nuclei_and_quantify_protein(input_dir, output_dir, to_plot=True, dapi_channel=dapi_channel,  protein_of_interest_1_channel=protein_of_interest_channel, protein_of_interest_1_name="ICAM1", protein_of_interest_2_channel=protein_of_interest_2_channel, protein_of_interest_2_name="VECad", phalloidin_channel=phalloidin_channel, max_proj=True)


LIF: Z:\Bel\Dorota_Image_Analzsis_Size_Spaces\Dorota_to_analzye_size_spaces.lif
Dorota_to_analzye_size_spaces_Fulv_TNF (9, 4, 2048, 2048) (1.9998575000000005, 0.283952125061065, 0.283952125061065)
Dorota_to_analzye_size_spaces_Fulv_E2P4_TNF (10, 4, 2048, 2048) (1.9998566666666668, 0.283952125061065, 0.283952125061065)
Dorota_to_analzye_size_spaces_ctrl (7, 4, 2048, 2048) (1.9998566666666602, 0.283952125061065, 0.283952125061065)
Dorota_to_analzye_size_spaces_E2P4 (9, 4, 2048, 2048) (1.9998574999999945, 0.283952125061065, 0.283952125061065)
Dorota_to_analzye_size_spaces_Fulv_TNF_2 (8, 4, 2048, 2048) (1.9998571428571426, 0.283952125061065, 0.283952125061065)
Dorota_to_analzye_size_spaces_Fulv_E2P4_TNF_2 (7, 4, 2048, 2048) (1.9998566666666668, 0.283952125061065, 0.283952125061065)
Dorota_to_analzye_size_spaces_Fulv_E2P4_TNF_3 (5, 4, 2048, 2048) (1.999857, 0.283952125061065, 0.283952125061065)
Dorota_to_analzye_size_spaces_Fulv_TNF_3 (6, 4, 2048, 2048) (1.9998569999999967, 0.28395212506106

Segmenting images:   0%|          | 0/32 [00:00<?, ?it/s]

Dorota_to_analzye_size_spaces_ctrl: 563 gaps, 40166 um^2 total, 22.0% of monolayer


Segmenting images:   3%|▎         | 1/32 [00:11<05:41, 11.00s/it]


KeyError: 'nucleus_label'

Dorota_to_analzye_size_spaces_E2P4: 25 gaps, 653 um^2 total, 0.2% of monolayer
Dorota_to_analzye_size_spaces_Fulv_TNF: 309 gaps, 19525 um^2 total, 5.8% of monolayer
Dorota_to_analzye_size_spaces_Fulv_E2P4_TNF: 240 gaps, 24003 um^2 total, 7.2% of monolayer
Dorota_to_analzye_size_spaces_Fulv_TNF_2: 156 gaps, 12021 um^2 total, 3.6% of monolayer
